In [1]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from matplotlib.lines import Line2D

from scipy.stats import pearsonr, spearmanr, kendalltau, norm
import statsmodels.api as sm
from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

## Part 2. Read data and define variables

这一块就是你说的真实数据导入部分。

In [2]:
# =========================================================
# 1. Read preprocessed data
# =========================================================

data_dir = Path("./preprocessed/preprocessed_with_zones")

classic_df = pd.read_csv(
    data_dir / "classic_with_climate_zones_filtered_30yr_mean.csv"
)

lpj_df = pd.read_csv(
    data_dir / "lpj_guess_with_climate_zones_filtered_30yr_mean.csv"
)


# =========================================================
# 2. Variable name mapping
# =========================================================

var_map = {
    "P": "precipitation",
    "T": "tran",
    "LAI": "lai",
    "ET": "evapotrans",
    "E_veg": "evspsblveg",
    "E_soil": "evspsblsoi"
}


# =========================================================
# 3. Input-output pairs
# =========================================================

pairs = [
    ("P", "ET"),
    ("P", "T"),
    ("P", "E_veg"),
    ("P", "E_soil"),
    ("LAI", "ET"),
    ("LAI", "T"),
    ("LAI", "E_veg"),
    ("LAI", "E_soil"),
]


# =========================================================
# 4. Climate zone settings
# =========================================================

zone_colors = {
    "HW": "#F27D7A",
    "HD": "#EFD447",
    "CW": "#95D98C",
    "CD": "#73AEEB",
}

zone_labels = {
    "HW": "WW - Warm-Wet",
    "HD": "WD - Warm-Dry",
    "CW": "CW - Cold-Wet",
    "CD": "CD - Cold-Dry",
}


# =========================================================
# 5. Axis labels
# =========================================================

full_label_map = {
    "P": "Precipitation (mm yr$^{-1}$)",
    "LAI": "Leaf Area Index (m$^2$ m$^{-2}$)",
    "ET": "Evapotranspiration (mm yr$^{-1}$)",
    "T": "Transpiration (mm yr$^{-1}$)",
    "E_veg": "Vegetation Evaporation (mm yr$^{-1}$)",
    "E_soil": "Soil Evaporation (mm yr$^{-1}$)"
}

## Part 3. Basic helper functions

In [3]:
def clean_xy(x, y):
    """
    Remove NaN / inf and sort by x.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) == 0:
        return x, y

    order = np.argsort(x)

    return x[order], y[order]

## Part 4. Different curve drawing / fitting methods

### 4.1 Equal-width bins

In [4]:
def fit_equal_width_bins(
    x,
    y,
    n_bins=20,
    min_count_per_bin=10
):
    """
    Bin x into equal-width intervals and compute mean x and mean y in each bin.
    """
    x, y = clean_xy(x, y)

    if len(x) < min_count_per_bin:
        return None

    x_min = np.nanmin(x)
    x_max = np.nanmax(x)

    if x_max == x_min:
        return None

    bins = np.linspace(x_min, x_max, n_bins + 1)
    bin_ids = np.digitize(x, bins) - 1

    xs = []
    ys = []
    counts = []

    for i in range(n_bins):
        if i == n_bins - 1:
            mask = (bin_ids == i) | (x == x_max)
        else:
            mask = bin_ids == i

        if np.sum(mask) >= min_count_per_bin:
            xs.append(np.nanmean(x[mask]))
            ys.append(np.nanmean(y[mask]))
            counts.append(np.sum(mask))

    if len(xs) < 5:
        return None

    return {
        "x": np.asarray(xs),
        "y": np.asarray(ys),
        "count": np.asarray(counts)
    }

### 4.2 Equal-count bins

In [5]:
def fit_equal_count_bins(
    x,
    y,
    n_bins=20,
    min_count_per_bin=10
):
    """
    Bin x by quantiles so that each bin contains a similar number of samples.
    """
    x, y = clean_xy(x, y)

    if len(x) < n_bins * min_count_per_bin:
        return None

    quantiles = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(x, quantiles)
    edges = np.unique(edges)

    if len(edges) < 5:
        return None

    xs = []
    ys = []
    counts = []

    for i in range(len(edges) - 1):
        if i == len(edges) - 2:
            mask = (x >= edges[i]) & (x <= edges[i + 1])
        else:
            mask = (x >= edges[i]) & (x < edges[i + 1])

        if np.sum(mask) >= min_count_per_bin:
            xs.append(np.nanmean(x[mask]))
            ys.append(np.nanmean(y[mask]))
            counts.append(np.sum(mask))

    if len(xs) < 5:
        return None

    return {
        "x": np.asarray(xs),
        "y": np.asarray(ys),
        "count": np.asarray(counts)
    }

### LOWESS

In [6]:
def fit_lowess_curve(
    x,
    y,
    frac=0.25
):
    """
    LOWESS smoothing curve.
    """
    x, y = clean_xy(x, y)

    if len(x) < 10:
        return None

    fitted = lowess(
        endog=y,
        exog=x,
        frac=frac,
        return_sorted=True
    )

    return {
        "x": fitted[:, 0],
        "y": fitted[:, 1]
    }

### 4.4 Polynomial regression

In [7]:
def fit_polynomial_curve(
    x,
    y,
    degree=3,
    n_grid=120
):
    """
    Polynomial regression fitted curve.
    """
    x, y = clean_xy(x, y)

    if len(x) < degree + 3:
        return None

    x_min = np.nanmin(x)
    x_max = np.nanmax(x)

    if x_max == x_min:
        return None

    x_norm = (x - x_min) / (x_max - x_min)

    x_grid = np.linspace(x_min, x_max, n_grid)
    x_grid_norm = (x_grid - x_min) / (x_max - x_min)

    poly = PolynomialFeatures(degree=degree, include_bias=True)

    X = poly.fit_transform(x_norm.reshape(-1, 1))
    X_grid = poly.transform(x_grid_norm.reshape(-1, 1))

    model = LinearRegression()
    model.fit(X, y)

    y_grid = model.predict(X_grid)

    return {
        "x": x_grid,
        "y": y_grid
    }

### 4.5 Piecewise linear regression

In [8]:
def fit_piecewise_linear_curve(
    x,
    y,
    min_side_n=30,
    n_grid=120
):
    """
    Two-segment piecewise linear fitting.
    The breakpoint is selected by minimum RSS.
    """
    x, y = clean_xy(x, y)

    if len(x) < 2 * min_side_n:
        return None

    best = None

    for idx in range(min_side_n, len(x) - min_side_n):
        x_left = x[:idx]
        y_left = y[:idx]

        x_right = x[idx:]
        y_right = y[idx:]

        X_left = sm.add_constant(x_left)
        X_right = sm.add_constant(x_right)

        left_model = sm.OLS(y_left, X_left).fit()
        right_model = sm.OLS(y_right, X_right).fit()

        rss = np.sum(left_model.resid ** 2) + np.sum(right_model.resid ** 2)

        if best is None or rss < best["rss"]:
            best = {
                "break_index": idx,
                "break_x": x[idx],
                "rss": rss,
                "left_model": left_model,
                "right_model": right_model
            }

    x_grid = np.linspace(np.nanmin(x), np.nanmax(x), n_grid)
    y_grid = np.empty_like(x_grid)

    left_mask = x_grid <= best["break_x"]
    right_mask = x_grid > best["break_x"]

    if np.sum(left_mask) > 0:
        y_grid[left_mask] = best["left_model"].predict(
            sm.add_constant(x_grid[left_mask])
        )

    if np.sum(right_mask) > 0:
        y_grid[right_mask] = best["right_model"].predict(
            sm.add_constant(x_grid[right_mask])
        )

    return {
        "x": x_grid,
        "y": y_grid,
        "break_x": best["break_x"]
    }

## Part 5. Build fitted curves for one relationship

这个函数的作用是：给定一个 raw x-y 关系，生成所有画线方法的 curve。

In [9]:
def fit_all_curve_methods(
    x,
    y,
    n_bins=20,
    min_count_per_bin=10,
    lowess_frac=0.25,
    poly_degree=3,
    piecewise_min_side_n=30
):
    curves = {}

    curves["Equal-width bins"] = fit_equal_width_bins(
        x,
        y,
        n_bins=n_bins,
        min_count_per_bin=min_count_per_bin
    )

    curves["Equal-count bins"] = fit_equal_count_bins(
        x,
        y,
        n_bins=n_bins,
        min_count_per_bin=min_count_per_bin
    )

    curves["LOWESS"] = fit_lowess_curve(
        x,
        y,
        frac=lowess_frac
    )

    curves["Polynomial degree 3"] = fit_polynomial_curve(
        x,
        y,
        degree=poly_degree
    )

    curves["Piecewise linear"] = fit_piecewise_linear_curve(
        x,
        y,
        min_side_n=piecewise_min_side_n
    )

    curves = {
        method: curve
        for method, curve in curves.items()
        if curve is not None
    }

    return curves

## Part 6. Build curve_results from real ESM data

这里是你现在真实数据最应该使用的结构。
最后会得到：

In [10]:
def build_curve_results_for_model(
    df,
    model_name,
    pairs,
    var_map,
    n_bins=20,
    min_count_per_bin=10,
    lowess_frac=0.25,
    poly_degree=3,
    piecewise_min_side_n=30
):
    curve_results = {}

    for x_label, y_label in pairs:
        x_col = var_map[x_label]
        y_col = var_map[y_label]

        if x_col not in df.columns or y_col not in df.columns:
            print(f"Missing columns for {model_name}: {x_col}, {y_col}")
            continue

        x_raw = df[x_col].to_numpy()
        y_raw = df[y_col].to_numpy()

        valid = np.isfinite(x_raw) & np.isfinite(y_raw)
        x_raw = x_raw[valid]
        y_raw = y_raw[valid]

        if len(x_raw) < 50:
            print(f"Insufficient data for {model_name}: {x_label} → {y_label}")
            continue

        curves = fit_all_curve_methods(
            x_raw,
            y_raw,
            n_bins=n_bins,
            min_count_per_bin=min_count_per_bin,
            lowess_frac=lowess_frac,
            poly_degree=poly_degree,
            piecewise_min_side_n=piecewise_min_side_n
        )

        curve_results[(x_label, y_label, model_name)] = {
            "x_raw": x_raw,
            "y_raw": y_raw,
            "curves": curves
        }

    return curve_results

In [11]:
classic_curve_results = build_curve_results_for_model(
    classic_df,
    model_name="CLASSIC",
    pairs=pairs,
    var_map=var_map,
    n_bins=20,
    min_count_per_bin=10,
    lowess_frac=0.25,
    poly_degree=3,
    piecewise_min_side_n=30
)

lpj_curve_results = build_curve_results_for_model(
    lpj_df,
    model_name="LPJ-GUESS",
    pairs=pairs,
    var_map=var_map,
    n_bins=20,
    min_count_per_bin=10,
    lowess_frac=0.25,
    poly_degree=3,
    piecewise_min_side_n=30
)

curve_results_all = {
    **classic_curve_results,
    **lpj_curve_results
}

KeyboardInterrupt: 

In [ ]:
curve_results_esoil = {
    key: value
    for key, value in curve_results_all.items()
    if key[1] == "E_soil"
}

## Part 7. Plot fitted curves from curve_results

这个先不做 formal test，只是把不同方法画出来。

In [ ]:
def plot_curves_from_results(
    curve_results,
    selected_y="E_soil",
    selected_xs=("P", "LAI"),
    selected_methods=None,
    n_cols=2,
    figsize_per_panel=(6.5, 5.0),
    scatter_alpha=0.15,
    scatter_size=6
):
    if selected_methods is None:
        selected_methods = [
            "Equal-width bins",
            "Equal-count bins",
            "LOWESS",
            "Polynomial degree 3",
            "Piecewise linear"
        ]

    selected_items = [
        (key, result)
        for key, result in curve_results.items()
        if key[1] == selected_y and key[0] in selected_xs
    ]

    n_panels = len(selected_items)
    n_rows = math.ceil(n_panels / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(figsize_per_panel[0] * n_cols, figsize_per_panel[1] * n_rows),
        dpi=160
    )

    axes = np.asarray(axes).reshape(-1)

    for i, (key, result) in enumerate(selected_items):
        x_label, y_label, model_name = key
        ax = axes[i]

        x_raw = result["x_raw"]
        y_raw = result["y_raw"]

        ax.scatter(
            x_raw,
            y_raw,
            s=scatter_size,
            alpha=scatter_alpha,
            color="0.55",
            linewidths=0
        )

        for method in selected_methods:
            if method not in result["curves"]:
                continue

            curve = result["curves"][method]

            ax.plot(
                curve["x"],
                curve["y"],
                linewidth=2,
                label=method
            )

        ax.set_title(
            f"{model_name}: {x_label} → {y_label}",
            fontsize=13
        )

        ax.set_xlabel(full_label_map.get(x_label, x_label), fontsize=11)
        ax.set_ylabel(full_label_map.get(y_label, y_label), fontsize=11)

        ax.grid(True, color="0.85", linewidth=0.7)
        ax.legend(fontsize=8, frameon=True)

    for j in range(n_panels, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_curves_from_results(
    curve_results_all,
    selected_y="E_soil",
    selected_xs=("P", "LAI")
)

## Part 8. Our curve-shape classifier

这一块才是“根据已经生成的曲线判定形状”。

In [ ]:
def classify_curve_shape(
    x_curve,
    y_curve,
    y_raw_std=None,
    flat_threshold=0.05,
    slope_tol=0.03,
    monotonic_fraction=0.70,
    min_net_change_ratio=0.10,
    turning_amplitude_ratio=0.10,
    side_fraction_threshold=0.60,
    roughness_max=3.0,
):
    x = np.asarray(x_curve, dtype=float)
    y = np.asarray(y_curve, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 5:
        return {"shape_type": "Insufficient data"}

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    unique_mask = np.concatenate([[True], np.diff(x) > 0])
    x = x[unique_mask]
    y = y[unique_mask]

    if len(x) < 5:
        return {"shape_type": "Insufficient data"}

    x_range = float(np.nanmax(x) - np.nanmin(x))
    y_range_raw = float(np.nanmax(y) - np.nanmin(y))

    if x_range == 0:
        return {"shape_type": "Insufficient data"}

    raw_scale_available = (
        y_raw_std is not None
        and np.isfinite(y_raw_std)
        and y_raw_std > 0
    )

    if raw_scale_available:
        y_range_to_raw_std = y_range_raw / y_raw_std

        if y_range_to_raw_std < flat_threshold:
            return {
                "shape_type": "Flat / negligible response",
                "y_range_raw": y_range_raw,
                "y_raw_std": y_raw_std,
                "y_range_to_raw_std": y_range_to_raw_std,
                "raw_scale_available": True,
            }

    elif y_range_raw == 0:
        return {
            "shape_type": "Flat / negligible response",
            "y_range_raw": 0.0,
            "y_raw_std": y_raw_std,
            "y_range_to_raw_std": np.nan,
            "raw_scale_available": False,
        }

    y_min = float(np.nanmin(y))
    y_range = float(np.nanmax(y) - y_min)

    if y_range == 0:
        return {
            "shape_type": "Flat / negligible response",
            "y_range_raw": y_range_raw,
            "y_raw_std": y_raw_std,
            "y_range_to_raw_std": (
                y_range_raw / y_raw_std if raw_scale_available else np.nan
            ),
            "raw_scale_available": raw_scale_available,
        }

    x_norm = (x - np.nanmin(x)) / x_range
    y_norm = (y - y_min) / y_range

    dx = np.diff(x_norm)
    dy = np.diff(y_norm)

    valid_dx = dx != 0
    dx = dx[valid_dx]
    dy = dy[valid_dx]

    if len(dx) < 4:
        return {"shape_type": "Insufficient data"}

    slopes = dy / dx

    signs = np.zeros_like(slopes, dtype=int)
    signs[slopes > slope_tol] = 1
    signs[slopes < -slope_tol] = -1
    signs[np.abs(slopes) <= slope_tol] = 0

    n_segments = len(signs)

    pos_fraction = float(np.sum(signs == 1) / n_segments)
    neg_fraction = float(np.sum(signs == -1) / n_segments)
    flat_fraction = float(np.sum(signs == 0) / n_segments)

    nonzero_signs = signs[signs != 0]

    if len(nonzero_signs) == 0:
        n_sign_changes = 0
        first_sign = 0
        last_sign = 0
    else:
        n_sign_changes = int(np.sum(nonzero_signs[1:] != nonzero_signs[:-1]))
        first_sign = int(nonzero_signs[0])
        last_sign = int(nonzero_signs[-1])

    net_change = float(y_norm[-1] - y_norm[0])
    net_change_ratio = abs(net_change)

    total_variation = float(np.sum(np.abs(np.diff(y_norm))))
    roughness_ratio = total_variation / (abs(net_change) + 1e-9)

    mostly_increasing = (
        net_change > min_net_change_ratio
        and pos_fraction + flat_fraction >= monotonic_fraction
    )

    mostly_decreasing = (
        net_change < -min_net_change_ratio
        and neg_fraction + flat_fraction >= monotonic_fraction
    )

    min_idx = int(np.argmin(y_norm))
    max_idx = int(np.argmax(y_norm))

    valley_is_internal = 0 < min_idx < len(y_norm) - 1
    peak_is_internal = 0 < max_idx < len(y_norm) - 1

    left_drop_to_min = float(y_norm[0] - y_norm[min_idx])
    right_rise_from_min = float(y_norm[-1] - y_norm[min_idx])

    left_rise_to_max = float(y_norm[max_idx] - y_norm[0])
    right_drop_from_max = float(y_norm[max_idx] - y_norm[-1])

    left_signs_to_min = signs[:max(min_idx, 1)]
    right_signs_from_min = signs[min_idx:]

    left_neg_or_flat = (
        float(np.mean((left_signs_to_min == -1) | (left_signs_to_min == 0)))
        if len(left_signs_to_min) > 0 else 0.0
    )

    right_pos_or_flat = (
        float(np.mean((right_signs_from_min == 1) | (right_signs_from_min == 0)))
        if len(right_signs_from_min) > 0 else 0.0
    )

    u_shape_candidate = (
        valley_is_internal
        and left_drop_to_min >= turning_amplitude_ratio
        and right_rise_from_min >= turning_amplitude_ratio
        and left_neg_or_flat >= side_fraction_threshold
        and right_pos_or_flat >= side_fraction_threshold
    )

    left_signs_to_max = signs[:max(max_idx, 1)]
    right_signs_from_max = signs[max_idx:]

    left_pos_or_flat = (
        float(np.mean((left_signs_to_max == 1) | (left_signs_to_max == 0)))
        if len(left_signs_to_max) > 0 else 0.0
    )

    right_neg_or_flat = (
        float(np.mean((right_signs_from_max == -1) | (right_signs_from_max == 0)))
        if len(right_signs_from_max) > 0 else 0.0
    )

    inverted_u_candidate = (
        peak_is_internal
        and left_rise_to_max >= turning_amplitude_ratio
        and right_drop_from_max >= turning_amplitude_ratio
        and left_pos_or_flat >= side_fraction_threshold
        and right_neg_or_flat >= side_fraction_threshold
    )

    if u_shape_candidate:
        shape_type = "U-shaped"
    elif inverted_u_candidate:
        shape_type = "Inverted-U-shaped"
    elif mostly_increasing:
        shape_type = "Monotonic increase"
    elif mostly_decreasing:
        shape_type = "Monotonic decrease"
    elif n_sign_changes >= 2 or roughness_ratio > roughness_max:
        shape_type = "Complex / uncertain nonlinear"
    else:
        shape_type = "Complex / uncertain nonlinear"

    return {
        "shape_type": shape_type,

        "y_range_raw": y_range_raw,
        "y_raw_std": y_raw_std,
        "y_range_to_raw_std": (
            y_range_raw / y_raw_std if raw_scale_available else np.nan
        ),
        "raw_scale_available": raw_scale_available,

        "net_change": net_change,
        "net_change_ratio": net_change_ratio,
        "pos_fraction": pos_fraction,
        "neg_fraction": neg_fraction,
        "flat_fraction": flat_fraction,
        "n_sign_changes": n_sign_changes,
        "first_sign": first_sign,
        "last_sign": last_sign,
        "roughness_ratio": roughness_ratio,

        "valley_is_internal": valley_is_internal,
        "peak_is_internal": peak_is_internal,
        "left_drop_to_min": left_drop_to_min,
        "right_rise_from_min": right_rise_from_min,
        "left_rise_to_max": left_rise_to_max,
        "right_drop_from_max": right_drop_from_max,
        "left_neg_or_flat": left_neg_or_flat,
        "right_pos_or_flat": right_pos_or_flat,
        "left_pos_or_flat": left_pos_or_flat,
        "right_neg_or_flat": right_neg_or_flat,
        "u_shape_candidate": u_shape_candidate,
        "inverted_u_candidate": inverted_u_candidate,
    }

## Part 9. Other shape / relationship tests

9.1 Pearson, Spearman, Kendall

In [ ]:
def run_correlation_tests(x, y):
    x, y = clean_xy(x, y)

    if len(x) < 5:
        return {
            "Pearson r": np.nan,
            "Pearson p": np.nan,
            "Spearman rho": np.nan,
            "Spearman p": np.nan,
            "Kendall tau": np.nan,
            "Kendall p": np.nan,
        }

    pearson_r, pearson_p = pearsonr(x, y)
    spearman_rho, spearman_p = spearmanr(x, y)
    kendall_tau, kendall_p = kendalltau(x, y)

    return {
        "Pearson r": pearson_r,
        "Pearson p": pearson_p,
        "Spearman rho": spearman_rho,
        "Spearman p": spearman_p,
        "Kendall tau": kendall_tau,
        "Kendall p": kendall_p,
    }

## 9.2 Lind-Mehlum quadratic test

In [ ]:
def lind_mehlum_quadratic_test(x, y, alpha=0.05):
    x, y = clean_xy(x, y)

    if len(x) < 10:
        return {"Lind_Mehlum": "Insufficient data"}

    x_min = np.nanmin(x)
    x_max = np.nanmax(x)

    if x_max == x_min:
        return {"Lind_Mehlum": "Insufficient data"}

    x_norm = (x - x_min) / (x_max - x_min)

    X = np.column_stack([
        np.ones_like(x_norm),
        x_norm,
        x_norm ** 2
    ])

    model = sm.OLS(y, X).fit()

    b0, b1, b2 = model.params
    cov = model.cov_params()

    x_low = np.nanmin(x_norm)
    x_high = np.nanmax(x_norm)

    slope_low = b1 + 2 * b2 * x_low
    slope_high = b1 + 2 * b2 * x_high

    c_low = np.array([0, 1, 2 * x_low])
    c_high = np.array([0, 1, 2 * x_high])

    se_low = np.sqrt(c_low @ cov @ c_low)
    se_high = np.sqrt(c_high @ cov @ c_high)

    z_low = slope_low / se_low if se_low > 0 else np.nan
    z_high = slope_high / se_high if se_high > 0 else np.nan

    p_low_neg = norm.cdf(z_low)
    p_high_pos = 1 - norm.cdf(z_high)

    p_low_pos = 1 - norm.cdf(z_low)
    p_high_neg = norm.cdf(z_high)

    p_u = max(p_low_neg, p_high_pos)
    p_inv_u = max(p_low_pos, p_high_neg)

    turning_point = -b1 / (2 * b2) if b2 != 0 else np.nan
    turning_inside = 0 < turning_point < 1

    u_supported = (
        b2 > 0
        and turning_inside
        and slope_low < 0
        and slope_high > 0
        and p_u < alpha
    )

    inv_u_supported = (
        b2 < 0
        and turning_inside
        and slope_low > 0
        and slope_high < 0
        and p_inv_u < alpha
    )

    if u_supported:
        result = "U-shaped"
    elif inv_u_supported:
        result = "Inverted-U-shaped"
    else:
        result = "No U-shape detected"

    return {
        "Lind_Mehlum": result,
        "LM_b2": b2,
        "LM_turning_point": turning_point,
        "LM_turning_inside": turning_inside,
        "LM_slope_low": slope_low,
        "LM_slope_high": slope_high,
        "LM_p_U": p_u,
        "LM_p_invU": p_inv_u,
    }

## 9.3 Two-line test

In [ ]:
def two_line_test(x, y, alpha=0.05, min_side_n=10):
    x, y = clean_xy(x, y)

    if len(x) < 2 * min_side_n:
        return {"Two_line": "Insufficient data"}

    best = None

    for idx in range(min_side_n, len(x) - min_side_n):
        x_left = x[:idx]
        y_left = y[:idx]

        x_right = x[idx:]
        y_right = y[idx:]

        X_left = sm.add_constant(x_left)
        X_right = sm.add_constant(x_right)

        left_model = sm.OLS(y_left, X_left).fit()
        right_model = sm.OLS(y_right, X_right).fit()

        rss = np.sum(left_model.resid ** 2) + np.sum(right_model.resid ** 2)

        if best is None or rss < best["rss"]:
            best = {
                "break_index": idx,
                "break_x": x[idx],
                "rss": rss,
                "left_model": left_model,
                "right_model": right_model
            }

    left_model = best["left_model"]
    right_model = best["right_model"]

    left_slope = left_model.params[1]
    right_slope = right_model.params[1]

    left_se = left_model.bse[1]
    right_se = right_model.bse[1]

    z_left = left_slope / left_se if left_se > 0 else np.nan
    z_right = right_slope / right_se if right_se > 0 else np.nan

    p_left_neg = norm.cdf(z_left)
    p_right_pos = 1 - norm.cdf(z_right)

    p_left_pos = 1 - norm.cdf(z_left)
    p_right_neg = norm.cdf(z_right)

    p_u = max(p_left_neg, p_right_pos)
    p_inv_u = max(p_left_pos, p_right_neg)

    u_supported = (
        left_slope < 0
        and right_slope > 0
        and p_u < alpha
    )

    inv_u_supported = (
        left_slope > 0
        and right_slope < 0
        and p_inv_u < alpha
    )

    if u_supported:
        result = "U-shaped"
    elif inv_u_supported:
        result = "Inverted-U-shaped"
    else:
        result = "No U-shape detected"

    return {
        "Two_line": result,
        "Two_line_break_x": best["break_x"],
        "Two_line_left_slope": left_slope,
        "Two_line_right_slope": right_slope,
        "Two_line_p_U": p_u,
        "Two_line_p_invU": p_inv_u,
    }

## 9.4 Order-restricted test

In [ ]:
def order_restricted_u_detection(
    x,
    y,
    min_side_n=10,
    amplitude_threshold=0.10
):
    x, y = clean_xy(x, y)

    if len(x) < 2 * min_side_n:
        return {"Order_restricted": "Insufficient data"}

    y_min = np.nanmin(y)
    y_max = np.nanmax(y)

    if y_max == y_min:
        return {"Order_restricted": "Flat / no U-shape detected"}

    y_norm = (y - y_min) / (y_max - y_min)

    best_u = None
    best_inv_u = None

    for idx in range(min_side_n, len(x) - min_side_n):
        x_left = x[:idx + 1]
        y_left = y_norm[:idx + 1]

        x_right = x[idx:]
        y_right = y_norm[idx:]

        iso_left_dec = IsotonicRegression(increasing=False, out_of_bounds="clip")
        iso_right_inc = IsotonicRegression(increasing=True, out_of_bounds="clip")

        yhat_left_u = iso_left_dec.fit_transform(x_left, y_left)
        yhat_right_u = iso_right_inc.fit_transform(x_right, y_right)

        rss_u = (
            np.sum((y_left - yhat_left_u) ** 2)
            + np.sum((y_right - yhat_right_u) ** 2)
        )

        left_drop = yhat_left_u[0] - yhat_left_u[-1]
        right_rise = yhat_right_u[-1] - yhat_right_u[0]

        if best_u is None or rss_u < best_u["rss"]:
            best_u = {
                "rss": rss_u,
                "turn_x": x[idx],
                "left_drop": left_drop,
                "right_rise": right_rise
            }

        iso_left_inc = IsotonicRegression(increasing=True, out_of_bounds="clip")
        iso_right_dec = IsotonicRegression(increasing=False, out_of_bounds="clip")

        yhat_left_inv = iso_left_inc.fit_transform(x_left, y_left)
        yhat_right_inv = iso_right_dec.fit_transform(x_right, y_right)

        rss_inv = (
            np.sum((y_left - yhat_left_inv) ** 2)
            + np.sum((y_right - yhat_right_inv) ** 2)
        )

        left_rise = yhat_left_inv[-1] - yhat_left_inv[0]
        right_drop = yhat_right_inv[0] - yhat_right_inv[-1]

        if best_inv_u is None or rss_inv < best_inv_u["rss"]:
            best_inv_u = {
                "rss": rss_inv,
                "turn_x": x[idx],
                "left_rise": left_rise,
                "right_drop": right_drop
            }

    u_valid = (
        best_u["left_drop"] >= amplitude_threshold
        and best_u["right_rise"] >= amplitude_threshold
    )

    inv_u_valid = (
        best_inv_u["left_rise"] >= amplitude_threshold
        and best_inv_u["right_drop"] >= amplitude_threshold
    )

    if u_valid and (not inv_u_valid or best_u["rss"] <= best_inv_u["rss"]):
        result = "U-shaped"
    elif inv_u_valid:
        result = "Inverted-U-shaped"
    else:
        result = "No U-shape detected"

    return {
        "Order_restricted": result,
        "Order_U_rss": best_u["rss"],
        "Order_U_turn_x": best_u["turn_x"],
        "Order_U_left_drop": best_u["left_drop"],
        "Order_U_right_rise": best_u["right_rise"],
        "Order_inv_rss": best_inv_u["rss"],
        "Order_inv_turn_x": best_inv_u["turn_x"],
        "Order_inv_left_rise": best_inv_u["left_rise"],
        "Order_inv_right_drop": best_inv_u["right_drop"],
    }

## Part 10. Run all shape tests on one curve

这个是统一接口。后面不管是 synthetic curve 还是真实 ESM fitted curve，都用它。

In [ ]:
def run_all_shape_tests_on_curve(
    x_curve,
    y_curve,
    y_raw_std=None,
    alpha=0.05,
    min_side_n=10,
    amplitude_threshold=0.10
):
    corr = run_correlation_tests(x_curve, y_curve)

    our = classify_curve_shape(
        x_curve,
        y_curve,
        y_raw_std=y_raw_std
    )

    lm = lind_mehlum_quadratic_test(
        x_curve,
        y_curve,
        alpha=alpha
    )

    tl = two_line_test(
        x_curve,
        y_curve,
        alpha=alpha,
        min_side_n=min_side_n
    )

    order_test = order_restricted_u_detection(
        x_curve,
        y_curve,
        min_side_n=min_side_n,
        amplitude_threshold=amplitude_threshold
    )

    return {
        **corr,

        "Our_method": our.get("shape_type", np.nan),
        "Lind_Mehlum": lm.get("Lind_Mehlum", np.nan),
        "Two_line": tl.get("Two_line", np.nan),
        "Order_restricted": order_test.get("Order_restricted", np.nan),

        "net_change": our.get("net_change", np.nan),
        "pos_fraction": our.get("pos_fraction", np.nan),
        "neg_fraction": our.get("neg_fraction", np.nan),
        "flat_fraction": our.get("flat_fraction", np.nan),
        "n_sign_changes": our.get("n_sign_changes", np.nan),
        "roughness_ratio": our.get("roughness_ratio", np.nan),
        "y_range_to_raw_std": our.get("y_range_to_raw_std", np.nan),

        **lm,
        **tl,
        **order_test,
    }

## Part 11. Run formal tests on existing curve_results

这才是你要的：根据现有 curve 去计算形状，打印表格。

In [ ]:
def run_formal_tests_on_curve_results(
    curve_results,
    alpha=0.05,
    min_side_n=10,
    amplitude_threshold=0.10
):
    records = []

    for key, result in curve_results.items():
        x_label, y_label, model_name = key

        x_raw = np.asarray(result["x_raw"], dtype=float)
        y_raw = np.asarray(result["y_raw"], dtype=float)

        y_raw_std = np.nanstd(y_raw)

        for method, curve in result["curves"].items():
            x_curve = curve["x"]
            y_curve = curve["y"]

            test_result = run_all_shape_tests_on_curve(
                x_curve,
                y_curve,
                y_raw_std=y_raw_std,
                alpha=alpha,
                min_side_n=min_side_n,
                amplitude_threshold=amplitude_threshold
            )

            model_short = "LPJ" if model_name == "LPJ-GUESS" else model_name

            relation_col = (
                f"{model_short}: {x_label} \u2192 {y_label}"
            )

            records.append({
                "model": model_name,
                "x": x_label,
                "y": y_label,
                "method": method,
                "Method": method,
                "relation_col": relation_col,
                **test_result
            })

    df = pd.DataFrame(records)

    num_cols = df.select_dtypes(include="number").columns
    df[num_cols] = df[num_cols].round(3)

    return df

In [ ]:
formal_df = run_formal_tests_on_curve_results(
    curve_results_all,
    alpha=0.05,
    min_side_n=10,
    amplitude_threshold=0.10
)

formal_df[
    [
        "model",
        "x",
        "y",
        "method",
        "Pearson r",
        "Spearman rho",
        "Kendall tau",
        "Our_method",
        "Lind_Mehlum",
        "Two_line",
        "Order_restricted"
    ]
]

## Part 12. E_soil only table

In [ ]:
formal_esoil_df = formal_df[
    formal_df["y"] == "E_soil"
].copy()

formal_esoil_df[
    [
        "model",
        "x",
        "y",
        "method",
        "Pearson r",
        "Spearman rho",
        "Kendall tau",
        "Our_method",
        "Lind_Mehlum",
        "Two_line",
        "Order_restricted"
    ]
]

### Part 13. Long table

In [ ]:
formal_long = formal_df.melt(
    id_vars=[
        "model",
        "x",
        "y",
        "method",
        "Method",
        "relation_col"
    ],
    value_vars=[
        "Pearson r",
        "Spearman rho",
        "Kendall tau",
        "Our_method",
        "Lind_Mehlum",
        "Two_line",
        "Order_restricted"
    ],
    var_name="Test",
    value_name="Result"
)

formal_long

## Part 16. Synthetic test data

这一部分是测试用的，和真实 ESM data 分开。

In [ ]:
def make_curve(func, x_range=(0, 1), n=120, noise=0.0, seed=0):
    rng = np.random.default_rng(seed)
    x = np.linspace(*x_range, n)
    y_true = func(x)
    y = y_true + noise * rng.standard_normal(n)
    return x, y, y_true


basic_test_cases = [
    ("Linear increase", lambda x: 0.5 * x, (0, 2), "Monotonic increase"),
    ("Linear decrease", lambda x: -0.5 * x, (0, 2), "Monotonic decrease"),

    ("Saturating increase", lambda x: 1 - np.exp(-3 * x), (0, 2), "Monotonic increase"),
    ("Saturating decrease", lambda x: np.exp(-3 * x), (0, 2), "Monotonic decrease"),

    ("U-shaped", lambda x: (x - 1.0) ** 2, (0, 2), "U-shaped"),
    ("Inverted-U-shaped", lambda x: 1.0 - (x - 1.0) ** 2, (0, 2), "Inverted-U-shaped"),

    ("Flat negligible response", lambda x: 0.001 * x, (0, 2), "Flat / negligible response"),
    ("Complex oscillation", lambda x: np.sin(3 * np.pi * x), (0, 2), "Complex / uncertain nonlinear"),
]


edge_test_cases = [
    ("Monotonic plus tiny early dip", lambda x: x - 0.005 * np.exp(-50 * (x - 0.05) ** 2), (0, 2), "Monotonic increase"),
    ("Nearly flat weak trend", lambda x: 0.01 * x, (0, 2), "Flat / negligible response"),
    ("Weak saturating increase but not flat", lambda x: 0.08 * (1 - np.exp(-3 * x)), (0, 2), "Monotonic increase"),
    ("Monotonic increase with small wiggles", lambda x: x + 0.03 * np.sin(8 * np.pi * x), (0, 2), "Monotonic increase"),

    ("Shallow U-shaped but meaningful", lambda x: 0.20 * (x - 1.0) ** 2, (0, 2), "U-shaped"),
    ("Very shallow U should be flat", lambda x: 0.02 * (x - 1.0) ** 2, (0, 2), "Flat / negligible response"),

    (
        "Asymmetric U-shaped",
        lambda x: np.where(
            x < 1.0,
            (x - 1.0) ** 2,
            0.35 * (x - 1.0) ** 2
        ),
        (0, 2),
        "U-shaped"
    ),

    ("Mild log saturation", lambda x: np.log1p(1.5 * x), (0, 2), "Monotonic increase"),
    ("Increasing with strong oscillation", lambda x: x + 0.20 * np.sin(6 * np.pi * x), (0, 2), "Complex / uncertain nonlinear"),

    ("Late acceleration increase", lambda x: x ** 2, (0, 2), "Monotonic increase"),
    ("Late acceleration decrease", lambda x: 1 - x ** 2, (0, 1), "Monotonic decrease"),
]

all_test_cases = basic_test_cases + edge_test_cases

## Part 17. Build synthetic curve results

这里让 synthetic data 也变成和真实 ESM 一样的 curve_results 结构。这样后面可以复用同一个 formal test 函数

In [ ]:
def build_curve_results_for_synthetic_tests(
    test_cases,
    noise=0.03,
    seed=42,
    n=300,
    n_bins=20,
    min_count_per_bin=10,
    lowess_frac=0.25,
    poly_degree=3,
    piecewise_min_side_n=30
):
    curve_results = {}
    expected_map = {}

    for test_name, func, x_range, expected in test_cases:
        x_raw, y_raw, y_true = make_curve(
            func,
            x_range=x_range,
            n=n,
            noise=noise,
            seed=seed
        )

        curves = fit_all_curve_methods(
            x_raw,
            y_raw,
            n_bins=n_bins,
            min_count_per_bin=min_count_per_bin,
            lowess_frac=lowess_frac,
            poly_degree=poly_degree,
            piecewise_min_side_n=piecewise_min_side_n
        )

        key = ("x", test_name, "Synthetic")

        curve_results[key] = {
            "x_raw": x_raw,
            "y_raw": y_raw,
            "y_true": y_true,
            "curves": curves
        }

        expected_map[key] = expected

    return curve_results, expected_map

In [ ]:
synthetic_curve_results, synthetic_expected_map = build_curve_results_for_synthetic_tests(
    all_test_cases,
    noise=0.03,
    seed=42,
    n=300,
    lowess_frac=0.25
)

## Part 18. Run formal tests on synthetic curves

In [ ]:
synthetic_formal_df = run_formal_tests_on_curve_results(
    synthetic_curve_results,
    alpha=0.05,
    min_side_n=10,
    amplitude_threshold=0.10
)

synthetic_formal_df["expected"] = synthetic_formal_df.apply(
    lambda row: synthetic_expected_map.get(
        ("x", row["y"], row["model"]),
        np.nan
    ),
    axis=1
)

synthetic_formal_df["Our_method_ok"] = (
    synthetic_formal_df["Our_method"] == synthetic_formal_df["expected"]
)

synthetic_formal_df[
    [
        "y",
        "expected",
        "method",
        "Pearson r",
        "Spearman rho",
        "Kendall tau",
        "Our_method",
        "Our_method_ok",
        "Lind_Mehlum",
        "Two_line",
        "Order_restricted"
    ]
]

In [ ]:
synthetic_accuracy = (
    synthetic_formal_df
    .groupby("method")["Our_method_ok"]
    .agg(["sum", "count"])
    .reset_index()
)

synthetic_accuracy["accuracy_percent"] = (
    synthetic_accuracy["sum"]
    / synthetic_accuracy["count"]
    * 100
).round(1)

synthetic_accuracy

def plot_synthetic_tests_with_labels(
    synthetic_curve_results,
    synthetic_expected_map,
    fit_method="LOWESS",
    n_cols=2,
    figsize_per_panel=(7.0, 5.8),
    bottom_text_y=-0.62,
    hspace=1.25,
    alpha=0.05
):
    items = list(synthetic_curve_results.items())

    n_cases = len(items)
    n_rows = math.ceil(n_cases / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(figsize_per_panel[0] * n_cols, figsize_per_panel[1] * n_rows),
        dpi=160
    )

    axes = np.asarray(axes).reshape(-1)

    records = []

    for i, (key, result) in enumerate(items):
        x_label, test_name, model_name = key
        ax = axes[i]

        x_raw = result["x_raw"]
        y_raw = result["y_raw"]
        y_true = result.get("y_true", None)

        expected = synthetic_expected_map[key]

        if fit_method not in result["curves"]:
            ax.axis("off")
            continue

        curve = result["curves"][fit_method]

        test_result = run_all_shape_tests_on_curve(
            curve["x"],
            curve["y"],
            y_raw_std=np.nanstd(y_raw),
            alpha=alpha
        )

        records.append({
            "test_name": test_name,
            "expected": expected,
            "fit_method": fit_method,
            **test_result
        })

        if y_true is not None:
            ax.plot(
                x_raw,
                y_true,
                linewidth=2.2,
                color="black",
                label="True curve"
            )

        ax.scatter(
            x_raw,
            y_raw,
            s=9,
            alpha=0.25,
            color="0.55",
            linewidths=0,
            label="Noisy points"
        )

        ax.plot(
            curve["x"],
            curve["y"],
            linewidth=2.0,
            color="tab:blue",
            label=fit_method
        )

        ok = test_result["Our_method"] == expected
        title_color = "black" if ok else "crimson"

        ax.set_title(
            test_name,
            fontsize=13,
            color=title_color,
            pad=8
        )

        ax.set_xlabel("x", fontsize=11.5)
        ax.set_ylabel("response", fontsize=11.5)

        bottom_text = (
            f"Expected: {expected}\n"
            f"Pearson r = {test_result['Pearson r']:.2f}; "
            f"Spearman ρ = {test_result['Spearman rho']:.2f}; "
            f"Kendall τ = {test_result['Kendall tau']:.2f}\n"
            f"Lind-Mehlum: {test_result['Lind_Mehlum']}\n"
            f"Two-line: {test_result['Two_line']}\n"
            f"Order-restricted: {test_result['Order_restricted']}\n"
            f"Our method: {test_result['Our_method']}"
        )

        ax.text(
            0.5,
            bottom_text_y,
            bottom_text,
            transform=ax.transAxes,
            ha="center",
            va="top",
            fontsize=9.5,
            linespacing=1.20,
            bbox=dict(
                boxstyle="round,pad=0.30",
                facecolor="white",
                edgecolor="0.70",
                alpha=0.96
            ),
            clip_on=False
        )

        ax.grid(True, color="0.85", linewidth=0.7)

        for side in ["top", "right", "bottom", "left"]:
            ax.spines[side].set_visible(True)
            ax.spines[side].set_linewidth(0.8)
            ax.spines[side].set_color("0.35")

        ax.tick_params(axis="both", labelsize=10.5)

    for j in range(n_cases, len(axes)):
        axes[j].axis("off")

    fig.suptitle(
        f"Synthetic curve-shape tests based on {fit_method}",
        fontsize=16,
        y=0.995
    )

    plt.tight_layout(rect=[0, 0.06, 1, 0.975])
    fig.subplots_adjust(wspace=0.20, hspace=hspace)

    plt.show()

    return pd.DataFrame(records)